# 03B — Add kinase-wide EGFR features

This notebook strengthens the baseline feature table by adding kinase-domain context.

It calculates, for every supported mutation–drug pair:

- distance to catalytic Lys745,
- distance to gatekeeper T790,
- distance to covalent-binding residue C797,
- distance to the DFG motif,
- distance to the αC helix,
- residue-contact graph degree,
- residue betweenness centrality,
- mean residue B-factor,
- distance to the ATP-pocket center,
- distance to the activation-loop center.

The output is:

`egfr_structural_features_enhanced.csv`

## Important scope

These are kinase-domain descriptors. This notebook does not attempt full-length EGFR modeling, molecular dynamics, or binding free-energy calculations.

## Step 1 — Upload two files

Upload:

1. `EGFR_mutant_structure_bundle.zip`
2. `egfr_structural_features.csv`


In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded))

## Step 2 — Install required packages

In [ ]:
!pip -q install pandas numpy biopython scipy networkx

## Step 3 — Extract the structure bundle and load the baseline feature table

In [ ]:
from pathlib import Path
import zipfile
import shutil
import pandas as pd

zip_name = "EGFR_mutant_structure_bundle.zip"
extract_dir = Path("EGFR_bundle")

if extract_dir.exists():
    shutil.rmtree(extract_dir)

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(extract_dir)

baseline = pd.read_csv("egfr_structural_features.csv")

print("Baseline rows:", len(baseline))
print("Baseline columns:", len(baseline.columns))
display(baseline.head())

## Step 4 — Define EGFR kinase landmarks

Canonical EGFR numbering is used:

- catalytic lysine: **K745**
- gatekeeper residue: **T790**
- covalent-binding residue: **C797**
- αC helix: residues **753–768**
- DFG motif: residues **855–857**
- activation loop: residues **855–884**
- ATP-pocket residues: a small representative set around the ATP/drug-binding site

The code only uses residues that are present in each PDB.

In [ ]:
LANDMARKS = {
    "lys745": [745],
    "t790": [790],
    "c797": [797],
    "alphaC": list(range(753, 769)),
    "DFG": [855, 856, 857],
    "activation_loop": list(range(855, 885)),
    "ATP_pocket": [718, 719, 726, 743, 745, 790, 793, 797, 800, 854, 855],
}

LANDMARKS

## Step 5 — Structural and graph helper functions

A residue-contact graph is built from Cα atoms:

- each residue is a node,
- an edge connects two residues whose Cα atoms are within 8 Å.

This gives interpretable graph descriptors without requiring a graph neural network.

In [ ]:
import numpy as np
import networkx as nx
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)

def load_first_model(pdb_path):
    structure = parser.get_structure("structure", str(pdb_path))
    return next(structure.get_models())

def protein_residue_map(model):
    mapping = {}
    for chain in model:
        for residue in chain:
            if residue.id[0] != " ":
                continue
            if "CA" not in residue:
                continue
            mapping[(chain.id, residue.id[1])] = residue
    return mapping

def choose_chain_for_position(residue_map, position, preferred_chain=None):
    if preferred_chain is not None and (preferred_chain, position) in residue_map:
        return preferred_chain

    matches = [chain for chain, pos in residue_map if pos == position]
    if len(matches) == 1:
        return matches[0]
    if not matches:
        return None
    return sorted(matches)[0]

def ca_coord(residue):
    return np.array(residue["CA"].coord, dtype=float)

def centroid_for_positions(residue_map, chain_id, positions):
    coords = []
    for pos in positions:
        residue = residue_map.get((chain_id, pos))
        if residue is not None and "CA" in residue:
            coords.append(ca_coord(residue))
    if not coords:
        return None
    return np.mean(coords, axis=0)

def point_distance(a, b):
    if a is None or b is None:
        return np.nan
    return float(np.linalg.norm(a - b))

def mean_residue_bfactor(residue):
    atoms = list(residue.get_atoms())
    if not atoms:
        return np.nan
    return float(np.mean([atom.get_bfactor() for atom in atoms]))

def build_ca_contact_graph(residue_map, cutoff=8.0):
    graph = nx.Graph()
    nodes = list(residue_map.keys())

    for node in nodes:
        graph.add_node(node)

    coords = np.array([ca_coord(residue_map[node]) for node in nodes])

    for i in range(len(nodes)):
        distances = np.linalg.norm(coords[i+1:] - coords[i], axis=1)
        neighbors = np.where(distances <= cutoff)[0]
        for offset in neighbors:
            j = i + 1 + int(offset)
            graph.add_edge(nodes[i], nodes[j])

    return graph

def residue_graph_features(graph, node):
    if node not in graph:
        return {
            "contact_graph_degree": np.nan,
            "betweenness_centrality": np.nan,
            "closeness_centrality": np.nan,
            "clustering_coefficient": np.nan,
        }

    betweenness = nx.betweenness_centrality(graph, normalized=True)
    closeness = nx.closeness_centrality(graph)
    clustering = nx.clustering(graph)

    return {
        "contact_graph_degree": float(graph.degree[node]),
        "betweenness_centrality": float(betweenness[node]),
        "closeness_centrality": float(closeness[node]),
        "clustering_coefficient": float(clustering[node]),
    }

## Step 6 — Precompute kinase graphs for the four reference structures

The graph is calculated once per drug-bound reference PDB and reused for all mutations associated with that structure.

In [ ]:
reference_dir = extract_dir / "reference_structures"

reference_cache = {}

for row in baseline[["drug", "pdb_id", "chain"]].drop_duplicates().itertuples(index=False):
    pdb_path = reference_dir / f"{row.pdb_id}.pdb"
    model = load_first_model(pdb_path)
    residue_map = protein_residue_map(model)
    graph = build_ca_contact_graph(residue_map, cutoff=8.0)

    reference_cache[(row.drug, row.pdb_id, row.chain)] = {
        "model": model,
        "residue_map": residue_map,
        "graph": graph,
    }

    print(
        f"{row.drug:12s} {row.pdb_id}: "
        f"{graph.number_of_nodes()} residues, "
        f"{graph.number_of_edges()} contacts"
    )

## Step 7 — Calculate the kinase-wide features

In [ ]:
enhanced_rows = []

for row in baseline.itertuples(index=False):
    key = (row.drug, row.pdb_id, row.chain)
    cached = reference_cache[key]

    residue_map = cached["residue_map"]
    graph = cached["graph"]

    mutation_node = (row.chain, int(row.position))
    mutation_residue = residue_map.get(mutation_node)

    if mutation_residue is None:
        raise KeyError(
            f"Mutation residue not found for {row.drug} {row.mutation}"
        )

    mutation_xyz = ca_coord(mutation_residue)

    lys745_xyz = centroid_for_positions(
        residue_map, row.chain, LANDMARKS["lys745"]
    )
    t790_xyz = centroid_for_positions(
        residue_map, row.chain, LANDMARKS["t790"]
    )
    c797_xyz = centroid_for_positions(
        residue_map, row.chain, LANDMARKS["c797"]
    )
    dfg_xyz = centroid_for_positions(
        residue_map, row.chain, LANDMARKS["DFG"]
    )
    alphaC_xyz = centroid_for_positions(
        residue_map, row.chain, LANDMARKS["alphaC"]
    )
    activation_loop_xyz = centroid_for_positions(
        residue_map, row.chain, LANDMARKS["activation_loop"]
    )
    atp_pocket_xyz = centroid_for_positions(
        residue_map, row.chain, LANDMARKS["ATP_pocket"]
    )

    graph_features = residue_graph_features(graph, mutation_node)

    enhanced_rows.append({
        "drug": row.drug,
        "pdb_id": row.pdb_id,
        "mutation": row.mutation,

        "distance_to_Lys745_A":
            point_distance(mutation_xyz, lys745_xyz),
        "distance_to_T790_A":
            point_distance(mutation_xyz, t790_xyz),
        "distance_to_C797_A":
            point_distance(mutation_xyz, c797_xyz),
        "distance_to_DFG_centroid_A":
            point_distance(mutation_xyz, dfg_xyz),
        "distance_to_alphaC_centroid_A":
            point_distance(mutation_xyz, alphaC_xyz),
        "distance_to_activation_loop_centroid_A":
            point_distance(mutation_xyz, activation_loop_xyz),
        "distance_to_ATP_pocket_centroid_A":
            point_distance(mutation_xyz, atp_pocket_xyz),

        "residue_mean_bfactor":
            mean_residue_bfactor(mutation_residue),

        **graph_features,
    })

kinase_features = pd.DataFrame(enhanced_rows)

print("Kinase-wide rows:", len(kinase_features))
print("New feature columns:", len(kinase_features.columns) - 3)
display(kinase_features.head(10))

## Step 8 — Merge the enhanced features with the baseline table

The merge key is:

- drug
- PDB structure
- mutation


In [ ]:
enhanced = baseline.merge(
    kinase_features,
    on=["drug", "pdb_id", "mutation"],
    how="left",
    validate="one_to_one",
)

assert len(enhanced) == len(baseline)

new_columns = [
    c for c in enhanced.columns
    if c not in baseline.columns
]

print("Enhanced rows:", len(enhanced))
print("Total columns:", len(enhanced.columns))
print("\nAdded columns:")
for col in new_columns:
    print(" -", col)

display(enhanced.head())

## Step 9 — Sanity checks

In [ ]:
print("Missing values among the new features:")
missing = enhanced[new_columns].isna().sum()
display(missing[missing > 0].sort_values(ascending=False))

print("\nFeature ranges:")
display(enhanced[new_columns].describe().T)

## Step 10 — Save and download the enhanced feature table

Use this CSV in the same model-training notebook.

The baseline file remains unchanged, which allows a direct baseline-versus-enhanced comparison.

In [ ]:
output_file = "egfr_structural_features_enhanced.csv"
enhanced.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Shape:", enhanced.shape)

files.download(output_file)

## Next step

Open Notebook 04 again and upload:

`egfr_structural_features_enhanced.csv`

In the first data-loading cell, either rename the uploaded file to `egfr_structural_features.csv`, or change:

```python
pd.read_csv("egfr_structural_features.csv")
```

to:

```python
pd.read_csv("egfr_structural_features_enhanced.csv")
```

Then rerun the grouped regression model and compare the new metrics with the baseline:

- baseline MAE ≈ 0.35
- baseline RMSE ≈ 0.47
- baseline R² ≈ -0.20
- baseline Spearman ≈ -0.03
